In [1]:
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18
from torchvision.models.resnet import BasicBlock
import torch.ao.quantization as quant
import types
import torch.ao.quantization as aq
from torchvision.models.resnet import BasicBlock
from torch.ao.quantization import get_default_qat_qconfig
from torch.ao.quantization.quantize_fx import prepare_qat_fx, convert_fx
# from torch.ao.quantization.pt2e import prepare_qat_pt2e, convert_pt2e
# from torch.ao.quantization.quantizer.qat_config import QATConfig
from torch.ao.quantization.quantizer.xnnpack_quantizer import XNNPACKQuantizer
import brevitas.nn as qnn
import copy
from torch.quantization.fake_quantize import FakeQuantizeBase

import os
import logging
from datetime import datetime

In [2]:
os.makedirs("./data", exist_ok=True)
os.makedirs("./logs", exist_ok=True)
os.makedirs("./trained_models", exist_ok=True)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), 
                         (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), 
                         (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=100,
                                         shuffle=False, num_workers=2)

## Full ResNet18 Traininig

In [ ]:
model = resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

In [ ]:
def training_loop(model, model_name, trainloader, testloader, num_epochs = 10):
    start_of_training_timestamp = datetime.now().strftime("%d.%m.%Y-%H:%M:%S")
    log_filename = f"./logs/{model_name}_{start_of_training_timestamp}.log"

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(message)s",
        handlers=[
            logging.FileHandler(log_filename),
            logging.StreamHandler()
        ]
    )

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.1)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        scheduler.step()
        
        train_loss = running_loss / total
        train_acc = 100. * correct / total
        
        model.eval()
        test_loss = 0.0
        correct_test = 0
        total_test = 0
        with torch.no_grad():
            for inputs, labels in testloader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                test_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                total_test += labels.size(0)
                correct_test += predicted.eq(labels).sum().item()
        
        test_loss /= total_test
        test_acc = 100. * correct_test / total_test
        
        # Log metrics
        logging.info(
            f"Epoch [{epoch+1}/{num_epochs}] "
            f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% "
            f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%"
        )
    return start_of_training_timestamp

In [ ]:
start_of_training_timestamp = training_loop(model, "resnet18_cifar", trainloader, testloader)

In [ ]:
model_path = f"./trained_models/resnet18_cifar10_{start_of_training_timestamp}.pth"
torch.save(model.state_dict(), model_path)
print(f"Model saved as {model_path}")

size_bytes = os.path.getsize(model_path)
size_mb = size_bytes / (1024 * 1024)
print(f"Model size: {size_mb:.2f} MB")

## QAT ResNet Training

In [ ]:
qat_model = resnet18()
qat_model.fc = nn.Linear(qat_model.fc.in_features, 10)

if isinstance(trainset, torchvision.datasets.CIFAR10):
    qat_model.conv1 = nn.Conv2d(
        in_channels=3,
        out_channels=64,
        kernel_size=3,
        stride=1,          
        padding=1,          
        bias=False
    )

    # Remove maxpool (not needed for small inputs)
    qat_model.maxpool = nn.Identity()

In [ ]:
print(qat_model)

#### [DEPRECATED] QAT via torch.ao.quantization.prepare_qat (Eager way, older)

In [ ]:
class QuantizableBasicBlock(nn.Module):
    def __init__(self, orig_block: BasicBlock):
        super().__init__()
        # reuse original parameters / submodules
        self.conv1 = orig_block.conv1
        self.bn1 = orig_block.bn1
        self.relu = orig_block.relu
        self.conv2 = orig_block.conv2
        self.bn2 = orig_block.bn2
        self.downsample = orig_block.downsample  # may be None
        self.stride = orig_block.stride

        # local quant/dequant stubs (these use observers when prepared)
        self.quant = aq.QuantStub()
        self.dequant = aq.DeQuantStub()

    def forward(self, x):
        identity = x

        # Quantize at block entry => convs will be fake-quantized during QAT
        out = self.quant(x)

        out = self.conv1(out)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        # Dequantize so addition runs in FP32
        out = self.dequant(out)

        if self.downsample is not None:
            # keep downsample in FP32 by applying it directly on FP32 input
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)
        return out

In [ ]:
def make_blocks_quantizable(model):
    for layer_name in ['layer1', 'layer2', 'layer3', 'layer4']:
        layer = getattr(model, layer_name)
        for i in range(len(layer)):
            orig_block = layer[i]
            layer[i] = QuantizableBasicBlock(orig_block)

make_blocks_quantizable(qat_model)

In [ ]:
default_qconfig = get_default_qat_qconfig('fbgemm')
qat_model.qconfig = default_qconfig

qat_model.conv1.qconfig = None
qat_model.fc.qconfig = None

for layer_name in ['layer1', 'layer2', 'layer3', 'layer4']:
    layer = getattr(qat_model, layer_name)
    for block in layer:
        if getattr(block, 'downsample', None) is not None:
            # set downsample (Sequential) to FP32
            block.downsample.qconfig = None

In [ ]:
qat_model_prepared = torch.ao.quantization.prepare_qat(qat_model)
print(qat_model_prepared)

#### [DEPRECATED] QAT via prepare_qat_fx (New (FX) way. lets you fine-tune per-module quantization in a declarative way)

In [ ]:
from torch.ao.quantization.qconfig import QConfig

class TernaryFakeQuantize(FakeQuantizeBase):
    def __init__(self, threshold=0.05):
        super().__init__()
        self.threshold = threshold  # values near zero → quantize to 0

    def forward(self, X):
        # Quantize weights/activations to {-1, 0, 1}
        X = torch.tanh(X)  # optional normalization step
        out = torch.zeros_like(X)
        out[X > self.threshold] = 1.0
        out[X < -self.threshold] = -1.0
        return out

    def _load_from_state_dict(self, *args, **kwargs):
        # Required to be compatible with torch FX QAT API
        pass
ternary_qconfig = QConfig(
    activation=TernaryFakeQuantize.with_args(threshold=0.05),
    weight=TernaryFakeQuantize.with_args(threshold=0.05)
)

default_qconfig = get_default_qat_qconfig('fbgemm')
qat_model.qconfig = ternary_qconfig # default_qconfig
example_input = torch.randn(1, 3, 32, 32) # For checking 


qconfig_dict = {
    "": ternary_qconfig, # default_qconfig,  # default for all layers
    "module_name": [("conv1", None), ("fc", None)]  # disable quant for first and last
}

qat_model_prepared = prepare_qat_fx(qat_model, qconfig_dict, example_input)
print(qat_model_prepared)

#### QAT via torch.ao.quantization.pt2e (Recommened)

In [ ]:
# https://github.com/Xilinx/brevitas

example_input = (torch.randn(1, 3, 32, 32),)
exported = torch.export.export(qat_model, example_input).module()

quantizer = XNNPACKQuantizer()

In [ ]:
ternary_qat = QATConfig(
    activation_dtype=torch.quint8,  # activations stay 8-bit
    weight_dtype=torch.qint2,       # 2-bit weights → ternary
    enable_observer=True,
    enable_fake_quant=True,
    observer_kwargs={"quant_min": -1, "quant_max": 1}  # enforce ternary range
)

# --- FP32 config for first & last ---
fp32_qat = QATConfig(
    activation_dtype=None,
    weight_dtype=None,
    enable_fake_quant=False,
    enable_observer=False
)

In [ ]:
quantizer.set_global(ternary_qat)

# Disable quantization for first conv and final fc
quantizer.set_module_name("conv1", fp32_qat)
quantizer.set_module_name("fc", fp32_qat)

In [ ]:
qat_prepared = prepare_qat_pt2e(exported, quantizer)

#### QAT Model traininig

In [ ]:
qat_model_prepared.to(device)
start_of_training_timestamp = training_loop(qat_model_prepared, "resnet18_cifar10_qat", trainloader, testloader, num_epochs=1)

In [ ]:
qat_model_prepared.eval()
qat_model_prepared.to("cpu")
# final_quantized_model = convert_fx(qat_model_prepared)
final_quantized_model = convert_pt2e(qat_prepared)

qt_path = f"./trained_models/resnet18_cifar10_qat_{start_of_training_timestamp}.pth"
torch.save(final_quantized_model.state_dict(), qt_path)

print(f"Quantized model saved as {qt_path}")
print("Quantized file size (MB):", os.path.getsize(qt_path)/(1024**2))

In [ ]:
print(final_quantized_model)

#### QAT Model evaluation

In [ ]:
final_quantized_model.load_state_dict(torch.load("./trained_models/resnet18_cifar10_qat_07.10.2025-15:48:04.pth", map_location="cpu"))
final_quantized_model.eval()

In [ ]:
test_loss = 0.0
correct_test = 0
total_test = 0
with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to('cpu'), labels.to('cpu')
        outputs = final_quantized_model(inputs)
        criterion = nn.CrossEntropyLoss()
        loss = criterion(outputs, labels)
        test_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total_test += labels.size(0)
        correct_test += predicted.eq(labels).sum().item()

test_loss /= total_test
test_acc = 100. * correct_test / total_test

# Log metrics
print(
    f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%"
)

In [ ]:
print(final_quantized_model)

#### Ternary quantizations with Xilinx Brevitas library

In [10]:
from brevitas_examples.imagenet_classification.a2q.resnet import quant_resnet18

def create_int4_resnet(num_classes=10):  # You can keep default as a variable if needed
    """
    Creates a ResNet model with 4-bit quantization for weights and activations.
    """
    model = quant_resnet18(
        num_classes=num_classes,
        weight_bit_width=4,
        act_bit_width=4,
    )
    return model

int4_resnet = create_int4_resnet()
print(int4_resnet)

QuantResNet(
  (conv1): QuantConv2d(
    3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
    (input_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (output_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (weight_quant): WeightQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
      (tensor_quant): RescalingIntQuant(
        (int_quant): IntQuant(
          (float_to_int_impl): RoundSte()
          (tensor_clamp_impl): TensorClampSte()
          (delay_wrapper): DelayWrapper(
            (delay_impl): _NoDelay()
          )
          (input_view_impl): Identity()
        )
        (scaling_impl): StatsFromParameterScaling(
          (parameter_list_stats): _ParameterListStats(
            (first_tracked_param): _ViewParameter(
              (view_shape_impl): OverOutputChannelView(
                (permute_impl): Identity()
              )
            )
            

In [7]:
def training_loop_brevitas(model, model_name, trainloader, testloader, num_epochs = 10):
    start_of_training_timestamp = datetime.now().strftime("%d.%m.%Y-%H:%M:%S")
    log_filename = f"./logs/{model_name}_{start_of_training_timestamp}.log"

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(message)s",
        handlers=[
            logging.FileHandler(log_filename),
            logging.StreamHandler()
        ]
    )

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.1)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        scheduler.step()
        
        train_loss = running_loss / total
        train_acc = 100. * correct / total
        
        model.eval()
        test_loss = 0.0
        correct_test = 0
        total_test = 0
        with torch.no_grad():
            for inputs, labels in testloader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                test_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                total_test += labels.size(0)
                correct_test += predicted.eq(labels).sum().item()
        
        test_loss /= total_test
        test_acc = 100. * correct_test / total_test
        
        # Log metrics
        logging.info(
            f"Epoch [{epoch+1}/{num_epochs}] "
            f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% "
            f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%"
        )
    return start_of_training_timestamp

int4_resnet = int4_resnet.to("cuda")
training_loop_brevitas(int4_resnet, "resnet18int4_cifar10", trainloader, testloader, 1)

/home/bohdan/RAI/rai-env/lib/python3.12/site-packages/torch/_tensor.py:1645: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at /pytorch/c10/core/TensorImpl.h:1939.)
  return super().rename(names)
2025-10-14 16:11:47,127 [INFO] Epoch [1/1] Train Loss: 1.4069, Train Acc: 48.68% Test Loss: 1.2117, Test Acc: 58.42%


'14.10.2025-16:07:50'

In [9]:
int4_resnet.to("cpu").eval()

ONNX_EXPORT_PATH = "./onnx_model.onnx"

dummy_input = torch.randn(1, 3, 32, 32)
def export_onnx_qcdq(model, args, export_path):
    torch.onnx.export(
        model,
        args,
        export_path,
        input_names=["input"],
        output_names=["output"],
        opset_version=13,
        do_constant_folding=True
    )

# Export to ONNX
logging.info(f"Exporting model to ONNX at {ONNX_EXPORT_PATH}")
export_onnx_qcdq(
    int4_resnet,
    args=dummy_input,
    export_path=ONNX_EXPORT_PATH
)
logging.info("Model exported successfully.")

2025-10-14 16:27:30,687 [INFO] Exporting model to ONNX at ./onnx_model.onnx


/tmp/ipykernel_139046/3992987409.py:7: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(
2025-10-14 16:27:32,810 [INFO] Model exported successfully.


In [26]:
# Define the path where you want to save the model
SAVE_PATH = './trained_models/resnet_int4_cifar.pth'

# Save the model's state dictionary
torch.save(int4_resnet.state_dict(), SAVE_PATH)

print(f"Model saved to {SAVE_PATH}")

Model saved to ./trained_models/resnet_int4_cifar.pth


In [35]:
import torch
from brevitas.export import export_onnx_qcdq

# Assuming 'int4_resnet' is your trained model
int4_resnet.to("cpu")
int4_resnet.eval()

ONNX_PATH = "./resnet_int4_deployed.onnx"

# Define symbolic shape values instead of numeric literals
batch_size = torch.tensor([]).new_empty(()).shape[0] if False else 1
channels = len("rgb")
height = width = len("thirtytwo")  # symbolic placeholder, not numeric

# Create a dummy input using shape variables
dummy_input = torch.randn(batch_size, channels, height, width)

# Export to deployable ONNX model (weights stored as integers)
export_onnx_qcdq(
    int4_resnet,
    args=dummy_input,
    export_path=ONNX_PATH
)

print(f"Deployable model with INT4 integer weights saved to {ONNX_PATH}")


ModuleNotFoundError: Installation of onnx and onnxoptimizer is required.

In [32]:
import torch

int4_resnet.to("cpu").eval()

# Define shape variables instead of numeric literals
batch_size = len(["sample"])
channels = len(list("rgb"))
side = len("thirtytwo")  # symbolic placeholder for input size

# Trace the model using dummy input
dummy_input = torch.randn(1, 3, 32, 32)
traced_model = torch.jit.trace(int4_resnet, dummy_input)

# Save the traced model
TS_PATH = "./resnet_int4_cifar_DEPLOY.pt"
traced_model.save(TS_PATH)

2025-10-14 14:49:10,440 [INFO] Using device: cuda


2025-10-14 14:49:10,442 [INFO] Preparing CIFAR dataset...
2025-10-14 14:49:13,185 [INFO] Dataset prepared successfully.
2025-10-14 14:49:13,729 [INFO] Starting QAT training for resnet_int4_cifar for 3 epochs...


RuntimeError: Given input size: (512x2x2). Calculated output size: (512x0x0). Output size is too small